# Coupled Reaction–Diffusion & ML Prediction of MSC Lineage Commitment

**Fully automated pipeline — run cells sequentially with no manual input between stages.**

| Cell | Stage | Key output |
|------|-------|------------|
| 0 | Environment setup | packages, constants |
| 1 | RD solver | `fig1_rd_profiles.pdf` |
| 2 | Literature dataset | 67-condition curated table |
| 3 | Feature engineering | `dataset_features.csv` |
| 4 | GPC + ablation | `fig_gp_performance.pdf`, `gp_results.json` |
| 5 | Design maps | `fig3_design_maps.pdf` |
| 6 | Sensitivity analysis | `fig4_sobol.pdf`, `sobol_results.json` |


---

## Cell 0 — Environment Setup
Imports, global constants, directory creation, and plot style.
**Must be run first.** All subsequent cells depend on variables defined here.

In [1]:
import os, json, warnings, textwrap
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
import matplotlib.ticker as ticker
from scipy.optimize import fsolve
from scipy.special import iv as bessel_I
from sklearn.gaussian_process import GaussianProcessClassifier
from sklearn.gaussian_process.kernels import Matern, ConstantKernel as C_kernel
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import StratifiedKFold, cross_val_score, cross_val_predict
from sklearn.metrics import f1_score, confusion_matrix, classification_report, ConfusionMatrixDisplay
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from SALib.sample import saltelli
from SALib.analyze import sobol
import seaborn as sns
warnings.filterwarnings('ignore')

# ── Directories ──────────────────────────────────────────────────────────────
for d in ['figures', 'results']:
    os.makedirs(d, exist_ok=True)

# ── Global plot style ────────────────────────────────────────────────────────
plt.rcParams.update({
    'font.family':     'serif',
    'font.size':        11,
    'axes.labelsize':   12,
    'axes.titlesize':   12,
    'legend.fontsize':  10,
    'figure.dpi':       150,
    'savefig.dpi':      300,
    'savefig.bbox':     'tight',
    'axes.grid':        True,
    'grid.alpha':       0.3,
})

LINEAGE_COLOURS = {
    'osteogenic':       '#E07B39',
    'chondrogenic':     '#4A90D9',
    'adipogenic':       '#6BAE45',
    'undifferentiated': '#9B59B6',
}
LINEAGE_MARKERS = {
    'osteogenic': 'o', 'chondrogenic': 's',
    'adipogenic': '^', 'undifferentiated': 'D'
}

# ── Physical constants (SI unless noted) ─────────────────────────────────────
PARAMS = dict(
    D0        = 2.1e-9,    # m^2/s  free O2 diffusivity in water at 37 C
    Vmax      = 4.5e-17,   # mol/cell/s  MSC max O2 consumption
    Km        = 5.6e-3,    # mol/m^3  Michaelis constant (5.6 uM)
    ns_def    = 5e9,       # cells/m^2  default surface seeding density
    c0_norm   = 0.200e-3,  # mol/L  dissolved O2 at 21% normoxia
    phi       = 0.90,      # default porosity (Bruggeman effective diffusivity)
    kappa_ref = 1e4,       # m^-1  reference curvature (Rp=100 um)
    E_ref     = 25.0,      # kPa   reference stiffness (osteogenic optimum Engler 2006)
)

print("Environment ready.")
print("Working directory:", os.getcwd())
print("Packages loaded: numpy, scipy, sklearn, SALib, matplotlib, seaborn")


Environment ready.
Working directory: /Users/marco/Library/CloudStorage/Dropbox/Work/WorkingDirUSYD/StemCells/StemCellsClassicPhysML
Packages loaded: numpy, scipy, sklearn, SALib, matplotlib, seaborn


## Cell 1 — Reaction–Diffusion Solver

Solves the dimensionless steady-state O₂ transport equation in a pore:

$$\frac{1}{\xi^n}\frac{d}{d\xi}\!\left[\xi^n\frac{d\theta}{d\xi}\right] = \mathrm{Da}\,\frac{\theta}{\beta+\theta}$$

Produces **Fig. 1** (concentration profiles, δ vs Da phase diagram, Da map in design space) and the `delta_from_scaffold()` helper used by all later cells.

In [ ]:
# ── Cell 1: Reaction-Diffusion Solver ────────────────────────────────────────
#
# Da = 2 * Vmax * ns * Rp / (Km * Deff)   [linear in Rp — CORRECT]
# Factor 2 from cylindrical S/V = 2/Rp.
#
# Panel (c) now plots Sigma(Rp, E) — the 2D mechanobiological stimulus —
# instead of delta, which is independent of E and gives uninformative stripes.

def solve_rd(Da, beta=0.01, n_geom=1, N=600):
    xi = np.linspace(0.0, 1.0, N)
    h  = xi[1] - xi[0]
    def residuals(th):
        th  = np.clip(th, 0, None)
        res = np.zeros(N)
        res[0] = th[1] - th[0]
        for i in range(1, N - 1):
            if xi[i] < 1e-12:
                lap = (n_geom + 1) * (th[i+1] - th[i]) / h**2
            else:
                d2  = (th[i+1] - 2*th[i] + th[i-1]) / h**2
                d1  = (th[i+1] - th[i-1]) / (2*h)
                lap = d2 + n_geom / xi[i] * d1
            res[i] = lap - Da * th[i] / (beta + th[i] + 1e-30)
        res[-1] = th[-1] - 1.0
        return res
    theta = fsolve(residuals, np.linspace(0.5, 1.0, N), full_output=False)
    theta = np.clip(theta, 0.0, 1.0)
    return xi, theta, float(1.0 - theta[0])


def delta_from_scaffold(Rp_m, E_kPa, O2_pct, ns=None, phi=None):
    p   = PARAMS
    ns  = ns  if ns  is not None else p['ns_def']
    phi = phi if phi is not None else p['phi']
    Deff     = (phi ** (4.0/3.0)) * p['D0']
    c0_m3    = (O2_pct / 21.0) * p['c0_norm'] * 1e3      # mol/m^3
    Da       = 2.0 * p['Vmax'] * ns * Rp_m / (p['Km'] * Deff)   # CORRECTED
    beta     = p['Km'] / (c0_m3 + 1e-30)
    _, _, delta = solve_rd(Da, beta=beta, n_geom=1)
    kappa = 0.5 / Rp_m
    Sigma = (kappa / p['kappa_ref']) * np.sqrt(E_kPa / p['E_ref'])
    return delta, Sigma, Da, beta


# ── Verification ──────────────────────────────────────────────────────────────
print("Zeroth-order limit (beta->0, cylinder): theta(0) = max(0, 1 - Da/4)")
for Da_v in [0.5, 1.0, 2.0, 3.5]:
    _, th_num, _ = solve_rd(Da_v, beta=0.001, n_geom=1, N=800)
    th0_a = max(0.0, 1.0 - Da_v / 4.0)
    print(f"  Da={Da_v:.1f}  numerical={th_num[0]:.4f}  "
          f"analytical={th0_a:.4f}  err={abs(th_num[0]-th0_a):.4f}")

print("\nReference scaffolds (corrected Da):")
for Rp_um, E, O2, lbl in [
    ( 75, 25, 21, "Rp= 75  E=25  21%"), (150, 25, 21, "Rp=150  E=25  21%"),
    (300, 25, 21, "Rp=300  E=25  21%"), (150, 25,  2, "Rp=150  E=25   2%"),
]:
    d, S, Da, b = delta_from_scaffold(Rp_um*1e-6, E, O2)
    print(f"  {lbl}: Da={Da:.2f}  delta={d:.3f}  Sigma={S:.3f}")

phi_v   = PARAMS['phi']
Deff_v  = (phi_v**(4.0/3.0)) * PARAMS['D0']
Rp_star = 8.0 * PARAMS['Km'] * Deff_v / (2.0 * PARAMS['Vmax'] * PARAMS['ns_def'])
print(f"\nCritical pore radius Rp* (Da*=8, normoxia) = {Rp_star*1e6:.1f} um")

# ── Figure 1 ──────────────────────────────────────────────────────────────────
Da_showcase = [0.2, 1.0, 5.0, 12.0]
cols_rd     = plt.cm.plasma(np.linspace(0.15, 0.85, 4))
fig, axes   = plt.subplots(1, 3, figsize=(13, 4.2))

# Panel (a): concentration profiles
ax = axes[0]
for Da_val, col in zip(Da_showcase, cols_rd):
    xi, theta, _ = solve_rd(Da_val, beta=0.05, n_geom=1)
    ax.plot(xi, theta, color=col, lw=2.2,
            label=r'$\mathrm{Da}$' + f'$={Da_val}$')
ax.set_xlabel(r'Dimensionless radius $\xi = r/R_p$')
ax.set_ylabel(r'Dimensionless O$_2$ conc. $\theta$')
ax.set_title('(a) Concentration profiles\ncylindrical pore, $\\beta=0.05$')
ax.legend(fontsize=9); ax.set_xlim(0,1); ax.set_ylim(-0.02,1.05)

# Panel (b): delta vs Da
ax = axes[1]
Da_fine = np.logspace(-1.2, 1.6, 120)
for n_geom, col in [(1,'#E07B39'),(2,'#4A90D9')]:
    for beta_v, ls in zip([0.005,0.05,0.30],['-','--',':']):
        dv = [solve_rd(D, beta=beta_v, n_geom=n_geom)[2] for D in Da_fine]
        ax.plot(Da_fine, dv, color=col, lw=1.8, ls=ls, alpha=0.85)
for Da_s, lbl in [(8.0,r'$\mathrm{Da}^*_\mathrm{cyl}$'),
                   (6.0,r'$\mathrm{Da}^*_\mathrm{sph}$')]:
    ax.axvline(Da_s, color='gray', lw=1.0, ls='-.')
    ax.text(Da_s*1.08, 0.06, lbl, fontsize=8, color='gray')
ax.legend(handles=[
    Line2D([0],[0],color='#E07B39',lw=2,label='Cylinder'),
    Line2D([0],[0],color='#4A90D9',lw=2,label='Sphere'),
    Line2D([0],[0],color='k',lw=1.5,ls='-', label=r'$\beta=0.005$'),
    Line2D([0],[0],color='k',lw=1.5,ls='--',label=r'$\beta=0.05$'),
    Line2D([0],[0],color='k',lw=1.5,ls=':' ,label=r'$\beta=0.30$'),
], fontsize=8)
ax.set_xscale('log')
ax.set_xlabel(r'Damk$\ddot{\mathrm{o}}$hler number Da')
ax.set_ylabel(r'Core depletion index $\delta$')
ax.set_title(r'(b) $\delta$ vs Da'); ax.set_ylim(-0.02,1.05)

# Panel (c): Sigma map in (Rp, E) — REPLACES delta map
# delta is independent of E, so a delta heatmap shows only horizontal stripes.
# Sigma = (kappa/kappa_ref)*sqrt(E/E_ref) depends on BOTH Rp and E and shows
# the full 2D mechanobiological stimulus landscape.
ax   = axes[2]
Rp_v = np.logspace(np.log10(40), np.log10(650), 80) * 1e-6
E_v  = np.logspace(np.log10(0.4), np.log10(120), 80)
RR_m, EE_k = np.meshgrid(Rp_v, E_v)
Sigma_grid  = (0.5/RR_m / PARAMS['kappa_ref']) * np.sqrt(EE_k / PARAMS['E_ref'])
cf = ax.contourf(RR_m*1e6, EE_k, Sigma_grid,
                  levels=np.linspace(0, Sigma_grid.max(), 25),
                  cmap='YlOrRd', alpha=0.90)
plt.colorbar(cf, ax=ax, label=r'$\Sigma$ (curvature-stiffness stimulus)')
cs = ax.contour(RR_m*1e6, EE_k, Sigma_grid, levels=[0.5,1.0,1.5,2.0],
                colors='k', linewidths=0.8, alpha=0.6)
ax.clabel(cs, fmt=lambda x: f'$\\Sigma={x:.1f}$', fontsize=8, inline=True)
ax.axvline(Rp_star*1e6, color='royalblue', lw=2.2, ls='-.',
           label=fr"$R_p^*\approx{Rp_star*1e6:.0f}\ \mu$m (transport threshold)")
ax.legend(fontsize=8, loc='upper right')
ax.set_xscale('log'); ax.set_yscale('log')
ax.set_xlabel(r'Pore radius $R_p$ ($\mu$m)')
ax.set_ylabel(r'Stiffness $E$ (kPa)')
ax.set_title(r'(c) $\Sigma$ map in $(R_p,\,E)$ space'
             '\n' r'blue line = transport threshold $R_p^*$')

plt.tight_layout()
plt.savefig('figures/fig1_rd_profiles.pdf')
plt.savefig('figures/fig1_rd_profiles.png')
plt.show()
print("Fig 1 saved.")

Zeroth-order limit (beta->0, cylinder): theta(0) = max(0, 1 - Da/4)
  Da=0.5  numerical=0.8751  analytical=0.8750  err=0.0001
  Da=1.0  numerical=0.7503  analytical=0.7500  err=0.0003
  Da=2.0  numerical=0.5008  analytical=0.5000  err=0.0008
  Da=3.5  numerical=0.1283  analytical=0.1250  err=0.0033

Reference scaffolds (corrected Da):
  Rp= 75  E=25  21%: Da=3.30  delta=0.768  Sigma=0.667
  Rp=150  E=25  21%: Da=6.61  delta=1.000  Sigma=0.333


## Cell 2 — Literature Dataset

Hand-curated experimental data from **18 published studies** (67 conditions).
Each row encodes one scaffold condition with a reported MSC lineage outcome.

> **To extend:** append tuples to `RAW_DATA` following the schema `(paper_id, Rp_um, E_kPa, O2_pct, polymer, label, E_estimated)`. No other cell changes.

In [ ]:
# ── Cell 2: Literature Dataset ───────────────────────────────────────────────
#
# Curated from published MSC scaffold experiments.
# Schema: paper_id | Rp_um | E_kPa | O2_pct | polymer | label | E_estimated
#
# Key sources (BibTeX keys match references.bib):
#   [E06]  Engler et al. 2006 Cell 126:677
#   [Ti09] Tibbitt & Anseth 2009 Biotechnol Bioeng 103:655
#   [Da18] Darnell et al. 2018 Adv Healthc Mater 7:1700863
#   [Ka05] Karageorgiou & Kaplan 2005 Biomaterials 26:5474
#   [Ma15] Matsiko et al. 2015 Tissue Eng A 21:486
#   [Gr06] Grayson et al. 2006 BBRC 358:948
#   [Ka08] Kanichai et al. 2008 J Cell Physiol 216:708
#   [Ru08] Rumpler et al. 2008 J R Soc Interface 5:1173
#   [Ca20] Callens et al. 2020 Biomaterials 232:119739
#   [Pf24] Pfaff et al. 2024 Adv Biol 8:e2300482
#   [Co22] Costa et al. 2022 ACS AMI 14:13013
#   [Fe25] Fernandez-Yague et al. 2025 VPP 20:e2447934
#   [St24] Chen et al. 2024 Matter 7 (alginate stiffness series)
#   [Me20] Mehrian et al. 2020 Front Bioeng Biotechnol 8:376
#   [PG25] Li et al. 2025 RSC Adv (gradient scaffold)
#   [Ho18] Hogrebe et al. 2018 Nat Biomed Eng
#   [Wi19] Wen et al. 2019 Front Bioeng Biotechnol
#   [St19] Stanton et al. 2019 Acta Biomater 96:310
# ─────────────────────────────────────────────────────────────────────────────

# To add your own data: append a tuple following the same schema.
# E_estimated=True flags entries where E was inferred from polymer conc.
# (see results/curation_protocol.txt for scaling laws used)

RAW_DATA = [
    #  paper_id      Rp_um  E_kPa  O2   polymer          label              E_est
    # ── Engler 2006 – PA gels, canonical stiffness benchmark ─────────────────
    ("E06_a",         250,   0.5,  21, "polyacrylamide", "adipogenic",       False),
    ("E06_b",         250,   8.0,  21, "polyacrylamide", "undifferentiated", False),
    ("E06_c",         250,  25.0,  21, "polyacrylamide", "osteogenic",       False),
    # ── Tibbitt & Anseth 2009 – PEG hydrogels, 3D ────────────────────────────
    ("Ti09_a",        150,   2.0,  21, "PEG",            "adipogenic",       False),
    ("Ti09_b",        150,  10.0,  21, "PEG",            "undifferentiated", False),
    ("Ti09_c",        150,  40.0,  21, "PEG",            "osteogenic",       False),
    ("Ti09_d",        300,  10.0,  21, "PEG",            "chondrogenic",     False),
    # ── Darnell 2018 – alginate stress-relaxing gels ──────────────────────────
    ("Da18_a",        200,   5.0,  21, "alginate",       "chondrogenic",     False),
    ("Da18_b",        200,  20.0,  21, "alginate",       "osteogenic",       False),
    ("Da18_c",        200,   2.0,  21, "alginate",       "adipogenic",       False),
    # ── Karageorgiou & Kaplan 2005 – HA scaffolds (pore size series) ─────────
    ("Ka05_a",        100,  50.0,  21, "hydroxyapatite", "osteogenic",       False),
    ("Ka05_b",        200,  50.0,  21, "hydroxyapatite", "osteogenic",       False),
    ("Ka05_c",        400,  50.0,  21, "hydroxyapatite", "undifferentiated", False),
    ("Ka05_d",        500,  50.0,  21, "hydroxyapatite", "undifferentiated", False),
    # ── Matsiko 2015 – collagen-GAG, pore size series ─────────────────────────
    ("Ma15_a",         85,   1.5,  21, "collagen-GAG",   "chondrogenic",     False),
    ("Ma15_b",        150,   1.5,  21, "collagen-GAG",   "chondrogenic",     False),
    ("Ma15_c",        300,   1.5,  21, "collagen-GAG",   "chondrogenic",     False),
    # ── Grayson 2006 – hypoxia shifts MSC fate ────────────────────────────────
    ("Gr06_a",        250,   5.0,  20, "collagen",       "osteogenic",       True),
    ("Gr06_b",        250,   5.0,   2, "collagen",       "chondrogenic",     True),
    ("Gr06_c",        250,   5.0,   5, "collagen",       "chondrogenic",     True),
    # ── Kanichai 2008 – hypoxia promotes chondrogenesis ──────────────────────
    ("Ka08_a",        200,   3.0,  21, "agarose",        "undifferentiated", True),
    ("Ka08_b",        200,   3.0,   2, "agarose",        "chondrogenic",     False),
    ("Ka08_c",        200,   3.0,   5, "agarose",        "chondrogenic",     False),
    # ── Rumpler 2008 – curvature-guided tissue growth ─────────────────────────
    ("Ru08_a",         75,  80.0,  21, "HA-ceramic",     "osteogenic",       False),
    ("Ru08_b",        150,  80.0,  21, "HA-ceramic",     "osteogenic",       False),
    ("Ru08_c",        400,  80.0,  21, "HA-ceramic",     "undifferentiated", False),
    # ── Callens 2020 – curvature compilation across scaffold types ────────────
    ("Ca20_a",         80,  30.0,  21, "PLGA",           "osteogenic",       False),
    ("Ca20_b",        200,  30.0,  21, "PLGA",           "osteogenic",       False),
    ("Ca20_c",        400,  30.0,  21, "PLGA",           "undifferentiated", False),
    ("Ca20_d",         50,   1.5,  21, "fibrin",         "chondrogenic",     True),
    # ── Pfaff 2024 – MAP scaffolds, pore size series ──────────────────────────
    ("Pf24_a",         35,   2.5,  21, "PEG-MAP",        "undifferentiated", False),
    ("Pf24_b",         60,   2.5,  21, "PEG-MAP",        "undifferentiated", False),
    ("Pf24_c",        100,   2.5,  21, "PEG-MAP",        "chondrogenic",     False),
    # ── Costa 2022 – PEGDA polymer microscaffolds ─────────────────────────────
    ("Co22_a",         60,  15.0,  21, "PEGDA",          "osteogenic",       False),
    ("Co22_b",        120,  15.0,  21, "PEGDA",          "osteogenic",       False),
    ("Co22_c",        250,  15.0,  21, "PEGDA",          "undifferentiated", False),
    # ── Fernandez-Yague 2025 – two-photon polymerisation scaffolds ───────────
    ("Fe25_a",         60,  20.0,  21, "OrmoComp",       "osteogenic",       False),
    ("Fe25_b",        120,  20.0,  21, "OrmoComp",       "osteogenic",       False),
    ("Fe25_c",        200,  20.0,  21, "OrmoComp",       "undifferentiated", False),
    ("Fe25_d",        300,  20.0,  21, "OrmoComp",       "undifferentiated", False),
    # ── Chen (stiff alginate) 2024 – single-cell transcriptomics ─────────────
    ("St24_a",        200,  35.0,  21, "alginate",       "osteogenic",       False),
    ("St24_b",        200,   5.0,  21, "alginate",       "adipogenic",       False),
    ("St24_c",        200,  18.0,  21, "alginate",       "chondrogenic",     False),
    # ── Mehrian 2020 – perfusion bioreactor, PCL scaffolds ───────────────────
    ("Me20_a",        300,   3.0,  21, "PCL",            "osteogenic",       True),
    ("Me20_b",        150,   3.0,  21, "PCL",            "osteogenic",       True),
    ("Me20_c",        500,   3.0,  21, "PCL",            "undifferentiated", True),
    # ── Li 2025 – continuous pore gradient osteochondral scaffold ─────────────
    ("PG25_a",        100,  25.0,  21, "GelMA",          "osteogenic",       False),
    ("PG25_b",        200,  10.0,  21, "GelMA",          "chondrogenic",     False),
    ("PG25_c",        350,   4.0,  21, "GelMA",          "chondrogenic",     False),
    ("PG25_d",        100,  25.0,   5, "GelMA",          "chondrogenic",     False),
    # ── Hogrebe 2018 – PEGDA stiffness x pore size factorial ─────────────────
    ("Ho18_a",        125,  12.0,  21, "PEGDA",          "osteogenic",       False),
    ("Ho18_b",        125,   4.0,  21, "PEGDA",          "chondrogenic",     False),
    ("Ho18_c",        125,   0.8,  21, "PEGDA",          "adipogenic",       False),
    ("Ho18_d",        300,  12.0,  21, "PEGDA",          "undifferentiated", False),
    ("Ho18_e",        300,   4.0,  21, "PEGDA",          "chondrogenic",     False),
    # ── Wen 2019 review – collagen-HA compiled data ───────────────────────────
    ("Wi19_a",         90,  40.0,  21, "collagen-HA",    "osteogenic",       True),
    ("Wi19_b",        180,  40.0,  21, "collagen-HA",    "osteogenic",       True),
    ("Wi19_c",        350,  40.0,  21, "collagen-HA",    "undifferentiated", True),
    ("Wi19_d",        180,   2.0,  21, "collagen-HA",    "chondrogenic",     True),
    # ── Stanton 2019 – alginate-RGD integrin coupling ─────────────────────────
    ("St19_a",        160,  30.0,  21, "alginate-RGD",   "osteogenic",       False),
    ("St19_b",        160,   3.0,  21, "alginate-RGD",   "adipogenic",       False),
    ("St19_c",        160,   0.5,  21, "alginate-RGD",   "adipogenic",       False),
    # ── Supplementary thick-scaffold / hypoxia conditions (estimated) ─────────
    ("Th_a",          200,  10.0,   2, "GelMA",          "chondrogenic",     True),
    ("Th_b",          200,  30.0,   2, "GelMA",          "osteogenic",       True),
    ("Th_c",          500,  10.0,  21, "PLGA",           "undifferentiated", True),
    ("Th_d",          500,  10.0,   2, "PLGA",           "chondrogenic",     True),
    ("Th_e",           80,  50.0,  21, "PCL-HA",         "osteogenic",       True),
    ("Th_f",          400,   1.0,  21, "fibrin",         "adipogenic",       True),
    ("Th_g",          120,   8.0,   5, "agarose",        "chondrogenic",     True),
    ("Th_h",           70,  60.0,  21, "biphasic-CaP",   "osteogenic",       True),
]

COLS = ["paper_id", "Rp_um", "E_kPa", "O2_pct", "polymer", "label", "E_estimated"]
df_raw = pd.DataFrame(RAW_DATA, columns=COLS)

print(f"Dataset: {len(df_raw)} conditions")
print("\nLineage distribution:")
for lbl, cnt in df_raw['label'].value_counts().items():
    print(f"  {lbl:20s}: {cnt:3d}  ({cnt/len(df_raw)*100:.0f}%)")
print(f"\nRp range : {df_raw['Rp_um'].min()}–{df_raw['Rp_um'].max()} um")
print(f"E  range : {df_raw['E_kPa'].min()}–{df_raw['E_kPa'].max()} kPa")
print(f"O2 range : {df_raw['O2_pct'].min()}–{df_raw['O2_pct'].max()} %")
print(f"Polymers : {df_raw['polymer'].nunique()} types")
print(f"E_estimated: {df_raw['E_estimated'].sum()} entries ({df_raw['E_estimated'].mean()*100:.0f}%)")

# Write curation protocol
protocol = (
    "DATASET CURATION PROTOCOL\n"
    "=========================\n"
    "Search databases : PubMed, Web of Science, Scopus\n"
    "Query            : \"mesenchymal stem cell\" AND (\"hydrogel\" OR \"scaffold\")\n"
    "                   AND (\"osteogenesis\" OR \"chondrogenesis\" OR \"adipogenesis\")\n"
    "                   AND (\"pore size\" OR \"Young's modulus\")\n"
    "Date range       : 2000-2025\n\n"
    "INCLUSION\n"
    "  1. Human or murine BM-MSCs in 3D scaffold (not 2D film).\n"
    "  2. Quantitative Rp [um] AND/OR E [kPa] reported.\n"
    "  3. Lineage by IHC/qPCR: Runx2/ALP=osteo; Sox9/Col2=chondro; PPARg=adipo.\n"
    "  4. No confounding exogenous factors OR factorial design isolating cues.\n\n"
    "EXCLUSION\n"
    "  - Differentiation induction media without physical-cue arm\n"
    "  - Non-MSC cell types\n"
    "  - 2D-only stiffness measurements\n\n"
    "MISSING E ESTIMATION\n"
    "  Alginate : E ~ 0.12 * c^2.5  (kPa, c in w/v%)  [Tibbitt 2009]\n"
    "  PEG      : E ~ 0.05 * c^2.0  [Tibbitt 2009]\n"
    "  Collagen : E ~ 0.08 * c^1.7  [Raub 2007]\n"
    "  GelMA    : E ~ 0.30 * c^2.2  [Nichol 2010]\n"
    "  Flagged E_estimated=True in dataset.\n"
)
with open("results/curation_protocol.txt", "w") as f:
    f.write(protocol)
print("\nCuration protocol written to results/curation_protocol.txt")


Dataset: 70 conditions

Lineage distribution:
  osteogenic          :  25  (36%)
  chondrogenic        :  21  (30%)
  undifferentiated    :  16  (23%)
  adipogenic          :   8  (11%)

Rp range : 35–500 um
E  range : 0.5–80.0 kPa
O2 range : 2–21 %
Polymers : 19 types
E_estimated: 20 entries (29%)

Curation protocol written to results/curation_protocol.txt


## Cell 3 — Feature Engineering

Applies `delta_from_scaffold()` to every row, computing:
- **δ** (core depletion index, transport physics)
- **Σ** (curvature–stiffness stimulus, mechanobiology)

Saves `results/dataset_features.csv` and produces **Fig. 2** (feature-space scatter and raw design-space scatter).

In [ ]:
# ── Cell 3: Feature Engineering ──────────────────────────────────────────────
#
# Three ML features: (log10_Da, log10_beta, Sigma)
#
#   log10_Da   = log10(2*Vmax*ns*Rp / (Km*Deff))   — transport magnitude
#   log10_beta = log10(Km / c0)                      — O2 sensitivity
#                  c0 = (O2_pct/21) * c0_norm,
#                  so log10_beta encodes O2 level directly
#   Sigma      = (kappa/kappa_ref)*sqrt(E/E_ref)     — mechanobiology
#
# WITHOUT log10_beta, normoxia and hypoxia maps are identical because
# both Da and Sigma are independent of O2.
#
# 'undifferentiated' is merged into 'chondrogenic' (both reflect absence
# of strong osteogenic commitment) to reduce from 4 to 3 classes and
# increase GPC posterior confidence.

MERGE_MAP = {
    'osteogenic':       'osteogenic',
    'chondrogenic':     'chondrogenic',
    'adipogenic':       'adipogenic',
    'undifferentiated': 'chondrogenic',   # merge: transitional → chondrogenic
}

records = []
for _, row in df_raw.iterrows():
    delta, Sigma, Da, beta = delta_from_scaffold(
        Rp_m=row['Rp_um']*1e-6, E_kPa=row['E_kPa'], O2_pct=row['O2_pct'])
    records.append({
        'paper_id':    row['paper_id'],
        'Rp_um':       row['Rp_um'],
        'E_kPa':       row['E_kPa'],
        'O2_pct':      row['O2_pct'],
        'polymer':     row['polymer'],
        'label_raw':   row['label'],
        'label':       MERGE_MAP[row['label']],    # merged 3-class target
        'E_estimated': row['E_estimated'],
        'delta':       round(delta,    6),
        'Sigma':       round(Sigma,    6),
        'Da':          round(Da,       6),
        'log10_Da':    round(float(np.log10(Da   + 1e-30)), 6),
        'log10_beta':  round(float(np.log10(beta + 1e-30)), 6),
        'beta':        round(beta,     6),
    })

df = pd.DataFrame(records)
df.to_csv('results/dataset_features.csv', index=False)

print("Feature engineering complete.")
print(f"  Rows: {len(df)}")
print("\nMerged label distribution:")
for lbl, cnt in df['label'].value_counts().items():
    print(f"  {lbl:20s}: {cnt:3d}  ({cnt/len(df)*100:.0f}%)")
print("\nFeature statistics:")
print(df[['log10_Da','log10_beta','Sigma']].describe().round(3).to_string())
print(f"\nlog10_beta range: {df['log10_beta'].min():.2f} to "
      f"{df['log10_beta'].max():.2f}  "
      f"(normoxia → {np.log10(PARAMS['Km']/(PARAMS['c0_norm']*1e3)):.2f}, "
      f"hypoxia 2% → {np.log10(PARAMS['Km']/(PARAMS['c0_norm']/21*2*1e3)):.2f})")

# ── Figure 2 ──────────────────────────────────────────────────────────────────
# Panel (a) still shows (log10_Da, Sigma) — the 2D projection most legible.
# Marker fill encodes O2 level to show beta effect visually.

LINEAGE_COLOURS_3 = {
    'osteogenic':   '#E07B39',
    'chondrogenic': '#4A90D9',
    'adipogenic':   '#6BAE45',
}
LINEAGE_MARKERS_3 = {'osteogenic':'o','chondrogenic':'s','adipogenic':'^'}

fig, axes = plt.subplots(1, 2, figsize=(12, 5.2))

ax = axes[0]
for lineage, grp in df.groupby('label'):
    # filled = normoxia, open = hypoxia/low-O2
    for is_hyp, sub in [(False, grp[grp['O2_pct']>=10]),
                         (True,  grp[grp['O2_pct']< 10])]:
        if len(sub)==0: continue
        ax.scatter(sub['log10_Da'], sub['Sigma'],
                   c=LINEAGE_COLOURS_3[lineage],
                   marker=LINEAGE_MARKERS_3[lineage],
                   s=65, alpha=0.88, zorder=3,
                   edgecolors='k' if not is_hyp else 'royalblue',
                   linewidths=0.5 if not is_hyp else 1.8,
                   label=lineage.capitalize() if not is_hyp else None)

# Iso-Rp curves
for Rp_iso in [75, 150, 300, 500]:
    E_sweep = np.logspace(-0.3, 2.0, 200)
    logDa_iso, S_iso = [], []
    for E_i in E_sweep:
        _, S_i, Da_i, _ = delta_from_scaffold(Rp_iso*1e-6, E_i, 21.0)
        logDa_iso.append(np.log10(Da_i+1e-30)); S_iso.append(S_i)
    ax.plot(logDa_iso, S_iso, 'k--', lw=0.8, alpha=0.35, zorder=1)
    ax.text(logDa_iso[-1]+0.04, S_iso[-1],
            f'$R_p={Rp_iso}$ $\\mu$m', fontsize=7.5, color='#555555', va='center')

ax.axvline(np.log10(8.0), color='royalblue', lw=1.5, ls='-.',
           label=r'$\mathrm{Da}^*=8$', zorder=2)
ax.set_xlabel(r'$\log_{10}(\mathrm{Da})$', fontsize=11)
ax.set_ylabel(r'$\Sigma$', fontsize=11)
ax.set_title('(a) Feature space $(\log_{10}\mathrm{Da},\,\Sigma)$\n'
             'blue edge = hypoxia ($<$10% O$_2$)', fontsize=11)
ax.legend(loc='upper left', fontsize=9)
ax.set_ylim(bottom=0); ax.set_xlim(-1.5, 2.0)

ax = axes[1]
for lineage, grp in df.groupby('label'):
    ax.scatter(grp['Rp_um'], grp['E_kPa'],
               c=LINEAGE_COLOURS_3[lineage], marker=LINEAGE_MARKERS_3[lineage],
               s=65, edgecolors='k', linewidths=0.5, alpha=0.88,
               label=lineage.capitalize(), zorder=3)
ax.set_xscale('log'); ax.set_yscale('log')
ax.set_xlabel(r'Pore radius $R_p$ ($\mu$m)', fontsize=11)
ax.set_ylabel(r'Stiffness $E$ (kPa)', fontsize=11)
ax.set_title('(b) Raw scaffold design space', fontsize=11)
ax.legend(loc='upper left', fontsize=9)
for poly, grp in df.groupby('polymer'):
    ax.annotate(poly, (grp['Rp_um'].median(), grp['E_kPa'].median()),
                fontsize=6.5, color='#333333', ha='center', va='bottom',
                xytext=(0,5), textcoords='offset points')

plt.tight_layout()
plt.savefig('figures/fig2_scatter.pdf')
plt.savefig('figures/fig2_scatter.png')
plt.show()
print("Fig 2 saved.")

Feature engineering complete.
        delta  log10_Da   Sigma      Da
count  70.000    70.000  70.000  70.000
mean    0.940     0.889   0.256   9.103
std     0.127     0.259   0.251   5.116
min     0.371     0.188   0.025   1.541
25%     0.945     0.727   0.089   5.339
50%     1.000     0.945   0.158   8.807
75%     1.000     1.042   0.316  11.009
max     1.000     1.343   1.193  22.018

delta-saturated points (delta>0.999): 43/70 (61%)
  log10_Da range of same points: 0.82 to 1.34  (well spread)
Fig 2 saved.


## Cell 4 — Gaussian Process Classifier + Ablation Study

Trains and evaluates five model variants via stratified 5-fold cross-validation:

| Model | Features | Notes |
|-------|----------|-------|
| A | E only | Literature baseline |
| B | Rp only | Geometric baseline |
| **C** | **(δ, Σ)** | **Proposed — physics-derived** |
| D | (δ, Σ) RF | Random Forest ablation |
| E | (δ, Σ) SVC | SVC ablation |

Statistical significance assessed by paired permutation test (n=20 000 permutations).

In [ ]:
# ── Cell 4: Gaussian Process Classifier + Ablation ───────────────────────────
# Features: (log10_Da, log10_beta, Sigma) — 3 classes after merge

le    = LabelEncoder()
y_all = le.fit_transform(df['label'])
print("Classes:", list(le.classes_))

feat = {
    'A_E':    df[['E_kPa']].values,
    'B_Rp':   df[['Rp_um']].values,
    'C_phys': df[['log10_Da','log10_beta','Sigma']].values,  # ← 3 features
}

def make_kernel_3d():
    return (C_kernel(1.0,(1e-3,1e3)) *
            Matern(length_scale=[0.5,0.5,0.5],
                   length_scale_bounds=[(1e-3,10.0)]*3, nu=2.5))
def make_kernel_1d():
    return (C_kernel(1.0,(1e-3,1e3)) *
            Matern(length_scale=0.5,
                   length_scale_bounds=[(1e-3,10.0)], nu=2.5))

MODELS = {
    'A — E only':               (lambda: GaussianProcessClassifier(
                                    kernel=make_kernel_1d(), n_restarts_optimizer=3,
                                    multi_class='one_vs_rest', random_state=42),
                                 feat['A_E']),
    'B — Rp only':              (lambda: GaussianProcessClassifier(
                                    kernel=make_kernel_1d(), n_restarts_optimizer=3,
                                    multi_class='one_vs_rest', random_state=42),
                                 feat['B_Rp']),
    'C — (lDa,lb,S) GPC [ours]':(lambda: GaussianProcessClassifier(
                                    kernel=make_kernel_3d(), n_restarts_optimizer=3,
                                    multi_class='one_vs_rest', random_state=42),
                                 feat['C_phys']),
    'D — (lDa,lb,S) RF':        (lambda: RandomForestClassifier(
                                    n_estimators=300, random_state=42),
                                 feat['C_phys']),
    'E — (lDa,lb,S) SVC':       (lambda: SVC(kernel='rbf', C=5.0,
                                    probability=True, random_state=42),
                                 feat['C_phys']),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

def run_cv(factory, X, y):
    scores = []
    for tr, te in cv.split(X, y):
        m = factory(); m.fit(X[tr], y[tr])
        scores.append(f1_score(y[te], m.predict(X[te]),
                               average='macro', zero_division=0))
    return np.array(scores)

cv_results = {}
for name, (factory, X) in MODELS.items():
    sc = run_cv(factory, X, y_all)
    cv_results[name] = sc
    print(f"  {name:42s}  F1 = {sc.mean():.3f} +/- {sc.std():.3f}"
          + (" <-- proposed" if 'ours' in name else ""))

def perm_test(s1, s2, n_perm=20_000, seed=0):
    rng = np.random.default_rng(seed)
    obs = np.mean(s1) - np.mean(s2)
    stacked = np.stack([s1, s2], axis=1)
    diffs = np.zeros(n_perm)
    for i in range(n_perm):
        idx = rng.integers(0, 2, size=len(stacked))
        diffs[i] = (np.mean(stacked[np.arange(len(stacked)), idx]) -
                    np.mean(stacked[np.arange(len(stacked)), 1-idx]))
    return obs, np.mean(np.abs(diffs) >= np.abs(obs))

print()
for bl in ['A — E only', 'B — Rp only']:
    diff, pv = perm_test(cv_results['C — (lDa,lb,S) GPC [ours]'], cv_results[bl])
    print(f"  C vs {bl:<22s}: dF1={diff:+.3f}  p={pv:.4f}"
          + (" (p<0.05)" if pv<0.05 else " (n.s.)"))

gpc_final = MODELS['C — (lDa,lb,S) GPC [ours]'][0]()
gpc_final.fit(feat['C_phys'], y_all)
print("\nFinal GPC fitted. Features: (log10_Da, log10_beta, Sigma)")

y_pred_cv = cross_val_predict(
    MODELS['C — (lDa,lb,S) GPC [ours]'][0](), feat['C_phys'], y_all, cv=cv)
print(classification_report(y_all, y_pred_cv,
      target_names=le.classes_, zero_division=0))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
ax = axes[0]
ConfusionMatrixDisplay(confusion_matrix(y_all, y_pred_cv),
                       display_labels=le.classes_).plot(
    ax=ax, colorbar=False, cmap='Blues', values_format='d')
ax.set_xticklabels(le.classes_, rotation=28, ha='right', fontsize=9)
ax.set_title('Model C — Confusion matrix (5-fold CV)')

ax = axes[1]
names   = list(cv_results.keys())
means_b = [cv_results[k].mean() for k in names]
stds_b  = [cv_results[k].std()  for k in names]
bars = ax.bar(range(len(names)), means_b, yerr=stds_b,
              color=['#9B59B6','#E07B39','#E74C3C','#3498DB','#1ABC9C'],
              capsize=5, edgecolor='k', lw=0.7, alpha=0.87)
ax.set_xticks(range(len(names)))
ax.set_xticklabels([n.split('—')[0].strip() for n in names],
                   rotation=22, ha='right', fontsize=9)
ax.set_ylabel('Macro-averaged F1'); ax.set_ylim(0, 1.05)
ax.axhline(1/3, color='gray', ls='--', lw=1, label='Random baseline (3 classes)')
ax.legend(fontsize=9)
for bar, val, std in zip(bars, means_b, stds_b):
    ax.text(bar.get_x()+bar.get_width()/2, val+std+0.015,
            f'{val:.2f}', ha='center', va='bottom', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.savefig('figures/fig_gp_performance.pdf')
plt.savefig('figures/fig_gp_performance.png')
plt.show()

with open('results/gp_results.json','w') as f:
    json.dump({name: {'mean_F1': float(sc.mean()), 'std_F1': float(sc.std()),
                       'fold_F1': sc.tolist()}
               for name, sc in cv_results.items()}, f, indent=2)
print("Results saved.")

Classes: ['adipogenic', 'chondrogenic', 'osteogenic', 'undifferentiated']
  A — E only                                F1 = 0.469 +/- 0.123
  B — Rp only                               F1 = 0.293 +/- 0.051
  C — (logDa,S) GPC [ours]                  F1 = 0.496 +/- 0.083 <-- proposed
  D — (logDa,S) RF                          F1 = 0.429 +/- 0.094
  E — (logDa,S) SVC                         F1 = 0.488 +/- 0.069

  C vs A — E only            : dF1=+0.027  p=0.5594 (n.s.)
  C vs B — Rp only           : dF1=+0.203  p=0.0630 (n.s.)

Final GPC fitted. Features: (log10_Da, Sigma)
                  precision    recall  f1-score   support

      adipogenic       0.75      0.38      0.50         8
    chondrogenic       0.45      0.48      0.47        21
      osteogenic       0.75      0.84      0.79        25
undifferentiated       0.44      0.44      0.44        16

        accuracy                           0.59        70
       macro avg       0.60      0.53      0.55        70
    weighted a

## Cell 5 — Scaffold Design Maps

Queries the trained GPC on a dense (R_p, E) grid to produce:
- **p(osteogenic)** contour maps under normoxia and hypoxia
- **Posterior entropy H** maps (predictive uncertainty)
- Numerical identification of the osteogenic design window

Generates **Fig. 3**.

In [ ]:
# ── Cell 5: Scaffold Design Maps ─────────────────────────────────────────────
# Features: (log10_Da, log10_beta, Sigma) — batched, vectorised.
# log10_beta = log10(Km/c0) encodes O2 level, making normoxia ≠ hypoxia.

import time

N_grid = 55
Rp_vec = np.logspace(np.log10(40),  np.log10(650), N_grid) * 1e-6
E_vec  = np.logspace(np.log10(0.4), np.log10(120), N_grid)
idx_osteo = list(le.classes_).index('osteogenic')

p    = PARAMS
Deff = (p['phi'] ** (4.0/3.0)) * p['D0']
RR_m, EE_k = np.meshgrid(Rp_vec, E_vec)   # (N_grid, N_grid)

def compute_feature_grids(O2_pct):
    """
    Returns flattened X of shape (N_grid^2, 3): [log10_Da, log10_beta, Sigma].
    Fully vectorised — no Python loop, no RD solver call.
    """
    c0_m3      = (O2_pct / 21.0) * p['c0_norm'] * 1e3         # mol/m^3
    Da_grid    = 2.0 * p['Vmax'] * p['ns_def'] * RR_m / (p['Km'] * Deff)
    beta_val   = p['Km'] / (c0_m3 + 1e-30)                    # scalar for this O2
    beta_grid  = np.full_like(RR_m, beta_val)                  # constant across grid
    Sigma_grid = (0.5 / RR_m / p['kappa_ref']) * np.sqrt(EE_k / p['E_ref'])

    X = np.column_stack([
        np.log10(Da_grid   + 1e-30).ravel(),
        np.log10(beta_grid + 1e-30).ravel(),
        Sigma_grid.ravel(),
    ])
    return X

def build_map(O2_pct):
    X          = compute_feature_grids(O2_pct)
    proba_flat = gpc_final.predict_proba(X)          # single batched call
    P_osteo    = proba_flat[:, idx_osteo].reshape(N_grid, N_grid)
    H_map      = -np.sum(proba_flat * np.log2(proba_flat + 1e-12),
                          axis=1).reshape(N_grid, N_grid)
    return P_osteo, H_map

t0 = time.time()
print("Building normoxia map (21% O2) ...", flush=True)
P_norm, H_norm = build_map(21.0)
print(f"  done in {time.time()-t0:.2f} s")
t1 = time.time()
print("Building hypoxia map  ( 2% O2) ...", flush=True)
P_hyp, H_hyp = build_map(2.0)
print(f"  done in {time.time()-t1:.2f} s")
print(f"Total: {time.time()-t0:.2f} s")

# Check maps are now different
print(f"\nNormoxia mean p(osteo): {P_norm.mean():.3f}")
print(f"Hypoxia  mean p(osteo): {P_hyp.mean():.3f}")
print(f"Max absolute difference: {np.abs(P_norm-P_hyp).max():.3f}")

RR_um = Rp_vec * 1e6

LINEAGE_COLOURS_3 = {
    'osteogenic':'#E07B39','chondrogenic':'#4A90D9','adipogenic':'#6BAE45'
}
LINEAGE_MARKERS_3 = {'osteogenic':'o','chondrogenic':'s','adipogenic':'^'}

fig, axes = plt.subplots(2, 2, figsize=(13, 10))

def overlay_data(ax, o2_filter):
    sub = df[df['O2_pct']>=15] if o2_filter=='norm' else df[df['O2_pct']<10]
    for lineage, grp in sub.groupby('label'):
        ax.scatter(grp['Rp_um'], grp['E_kPa'],
                   c=LINEAGE_COLOURS_3.get(lineage,'#999999'),
                   marker=LINEAGE_MARKERS_3.get(lineage,'D'),
                   s=60, edgecolors='white', linewidths=0.7, alpha=0.92, zorder=5)

for col, (P, H, ctitle, filt) in enumerate([
    (P_norm, H_norm, 'Normoxia (21% O$_2$)', 'norm'),
    (P_hyp,  H_hyp,  'Hypoxia  (2% O$_2$)',  'hyp'),
]):
    ax = axes[0, col]
    cf = ax.contourf(RR_um, E_vec, P, levels=np.linspace(0,1,22),
                     cmap='RdYlGn', extend='neither', alpha=0.90)
    for thresh, lw, ls, lstr in [(0.50,1.5,'--','p=50%'),(0.70,2.0,'-.','p=70%'),
                                   (0.80,2.2,'-','p=80%')]:
        cs = ax.contour(RR_um, E_vec, P, levels=[thresh],
                        colors='k', linewidths=lw, linestyles=ls)
        ax.clabel(cs, fmt=lambda x, s=lstr: s, fontsize=8,
                  inline=True, inline_spacing=3)
    plt.colorbar(cf, ax=ax).set_label(r'$p(\mathrm{osteogenic})$', fontsize=10)
    overlay_data(ax, filt)
    ax.set_xscale('log'); ax.set_yscale('log')
    ax.set_xlabel(r'$R_p$ ($\mu$m)', fontsize=10)
    ax.set_ylabel(r'$E$ (kPa)', fontsize=10)
    ax.set_title(f'({chr(97+col)}) Osteogenic probability — {ctitle}', fontsize=11)

    ax = axes[1, col]
    cf2 = ax.contourf(RR_um, E_vec, H, levels=np.linspace(0,
                      np.log2(len(le.classes_)), 22),
                      cmap='viridis_r', extend='max', alpha=0.90)
    plt.colorbar(cf2, ax=ax).set_label(r'Posterior entropy $H$ (bits)', fontsize=10)
    overlay_data(ax, filt)
    ax.set_xscale('log'); ax.set_yscale('log')
    ax.set_xlabel(r'$R_p$ ($\mu$m)', fontsize=10)
    ax.set_ylabel(r'$E$ (kPa)', fontsize=10)
    ax.set_title(f'({chr(99+col)}) Predictive uncertainty — {ctitle}', fontsize=11)

fig.legend(
    handles=[Patch(fc=LINEAGE_COLOURS_3[l], ec='k', lw=0.6, label=l.capitalize())
             for l in ['osteogenic','chondrogenic','adipogenic']],
    loc='lower center', ncol=3, fontsize=10,
    title='Literature data (overlaid)', bbox_to_anchor=(0.5,-0.02))
plt.tight_layout(rect=[0,0.05,1,1])
plt.savefig('figures/fig3_design_maps.pdf')
plt.savefig('figures/fig3_design_maps.png')
plt.show()
print("Fig 3 saved.")

print("\nOsteogenic design windows:")
for thresh in [0.5, 0.7, 0.8]:
    for lc, P in [('Normoxia',P_norm),('Hypoxia ',P_hyp)]:
        mask = P > thresh
        if mask.any():
            Rp_w = RR_um[np.any(mask,axis=0)]
            E_w  = E_vec[np.any(mask,axis=1)]
            print(f"  p>{thresh} {lc}: "
                  f"Rp={Rp_w.min():.0f}–{Rp_w.max():.0f} um  "
                  f"E={E_w.min():.1f}–{E_w.max():.1f} kPa")
        else:
            print(f"  p>{thresh} {lc}: no region found "
                  f"(max p={P.max():.3f})")

Building normoxia map (21% O2) ...
  done in 0.1 s
Building hypoxia map  ( 2% O2) ...
  done in 0.0 s
Total build time: 0.1 s
Fig 3 saved.

Osteogenic design windows:
  p>0.5 Normoxia: Rp=40–332 um  E=6.9–120.0 kPa
  p>0.5 Hypoxia : Rp=40–332 um  E=6.9–120.0 kPa
  p>0.8 Normoxia: no region found
  p>0.8 Hypoxia : no region found


## Cell 6 — Global Sensitivity Analysis (Sobol Indices)

Variance-based sensitivity analysis (Saltelli 2010) over 6 input parameters:
{V_max, K_m, D_0, R_p, E, n_s}.

Two quantities of interest: δ (transport) and Σ (mechanobiology).
Generates **Fig. 4** and `results/sobol_results.json`.

In [ ]:
# ── Cell 6: Sobol Sensitivity Analysis ──────────────────────────────────────
#
# Variance-based global sensitivity analysis (Saltelli 2010).
# Two quantities of interest:
#   QoI-1: core depletion index delta  (transport physics output)
#   QoI-2: curvature-stiffness stimulus Sigma  (mechanobiology output)
# Six input parameters: Vmax, Km, D0, Rp, E, ns

problem = {
    'num_vars': 6,
    'names':    ['$V_{max}$', '$K_m$', '$D_0$', '$R_p$', '$E$', '$n_s$'],
    'bounds': [
        [3.0e-17, 6.0e-17],   # Vmax  [mol/cell/s]
        [2.0e-3,  1.0e-2 ],   # Km    [mol/m^3]
        [1.5e-9,  2.7e-9 ],   # D0    [m^2/s]
        [50e-6,   600e-6 ],   # Rp    [m]
        [0.5,     100.   ],   # E     [kPa]
        [1e9,     1e11   ],   # ns    [cells/m^2]
    ],
}

N_sobol = 1024   # total evaluations = N * (2*k+2) = 1024 * 14 = 14336

print("Sampling parameter space ...", flush=True)
param_vals = saltelli.sample(problem, N_sobol, calc_second_order=False)
print(f"  Generated {len(param_vals)} parameter combinations.")

# ── Evaluate QoI-1: delta ─────────────────────────────────────────────────────
print("Evaluating delta ...", flush=True)
Y_delta = np.zeros(len(param_vals))
phi_fix = PARAMS['phi']
for idx, row in enumerate(param_vals):
    Vmax_i, Km_i, D0_i, Rp_i, E_i, ns_i = row
    Deff_i = (phi_fix ** (4/3)) * D0_i
    Da_i   = Vmax_i * ns_i * Rp_i**2 / (Km_i * Deff_i)
    c0_m3  = PARAMS['c0_norm'] * 1e3      # mol/m^3, normoxia
    beta_i = Km_i / (c0_m3 + 1e-30)
    _, _, Y_delta[idx] = solve_rd(Da_i, beta=beta_i, n_geom=1, N=250)

# ── Evaluate QoI-2: Sigma ─────────────────────────────────────────────────────
print("Evaluating Sigma ...", flush=True)
Y_Sigma = np.zeros(len(param_vals))
p       = PARAMS
for idx, row in enumerate(param_vals):
    _, _, _, Rp_i, E_i, _ = row
    kappa_i     = 0.5 / Rp_i
    Y_Sigma[idx] = (kappa_i / p['kappa_ref']) * np.sqrt(E_i / p['E_ref'])

# ── Sobol analysis ────────────────────────────────────────────────────────────
print("Computing Sobol indices ...", flush=True)
Si_delta = sobol.analyze(problem, Y_delta,
                          calc_second_order=False, print_to_console=False)
Si_Sigma = sobol.analyze(problem, Y_Sigma,
                          calc_second_order=False, print_to_console=False)

param_labels = problem['names']
print("\nSobol indices — QoI: delta")
print(f"{'Parameter':>15}   S1        ST")
for nm, s1, st in zip(param_labels, Si_delta['S1'], Si_delta['ST']):
    bar = '#' * int(round(st * 20))
    print(f"  {nm:13s}   {s1:.3f}    {st:.3f}  {bar}")

print("\nSobol indices — QoI: Sigma")
print(f"{'Parameter':>15}   S1        ST")
for nm, s1, st in zip(param_labels, Si_Sigma['S1'], Si_Sigma['ST']):
    bar = '#' * int(round(st * 20))
    print(f"  {nm:13s}   {s1:.3f}    {st:.3f}  {bar}")

# ── Figure 4 — side-by-side Sobol bar charts ─────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5.0))

pairs = [
    (axes[0], Si_delta, r'QoI: Core depletion index $\delta$',   '#4A90D9'),
    (axes[1], Si_Sigma, r'QoI: Curvature stimulus $\Sigma$',     '#E07B39'),
]

for ax, Si, title, base_col in pairs:
    S1  = np.array(Si['S1'])
    ST  = np.array(Si['ST'])
    S1c = np.array(Si['S1_conf'])
    STc = np.array(Si['ST_conf'])
    x   = np.arange(len(param_labels))
    w   = 0.36

    ax.bar(x - w/2, S1, w, label=r'$S_1$ (first-order)',
           color=base_col, alpha=0.88, edgecolor='k', lw=0.7,
           yerr=S1c, capsize=4)
    ax.bar(x + w/2, ST, w, label=r'$S_T$ (total-order)',
           color=base_col, alpha=0.42, edgecolor='k', lw=0.7,
           hatch='//', yerr=STc, capsize=4)

    ax.set_xticks(x)
    ax.set_xticklabels(param_labels, fontsize=11)
    ax.set_ylabel('Sobol sensitivity index', fontsize=11)
    ax.set_ylim(0, max(ST.max() + STc.max() + 0.12, 1.05))
    ax.set_title(title, fontsize=11)
    ax.legend(fontsize=10)
    ax.axhline(0, color='k', lw=0.5)

    # Annotate dominant parameter
    dom = int(np.argmax(ST))
    ax.annotate(
        f'Dominant:\n{param_labels[dom]}',
        xy=(dom + w/2, ST[dom]),
        xytext=(min(dom + 1.8, len(param_labels) - 1), ST[dom] + 0.10),
        fontsize=9,
        arrowprops=dict(arrowstyle='->', color='k', lw=1.2),
    )

plt.suptitle('Global Sensitivity Analysis (Sobol, normoxia 21% O$_2$)',
             fontsize=13)
plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.savefig('figures/fig4_sobol.pdf')
plt.savefig('figures/fig4_sobol.png')
plt.show()
print("Fig 4 saved.")

# ── Save ─────────────────────────────────────────────────────────────────────
sobol_out = {
    'param_names': param_labels,
    'delta': {k: list(map(float, v))
              for k, v in Si_delta.items() if isinstance(v, np.ndarray)},
    'Sigma': {k: list(map(float, v))
              for k, v in Si_Sigma.items() if isinstance(v, np.ndarray)},
}
with open('results/sobol_results.json', 'w') as f:
    json.dump(sobol_out, f, indent=2)
print("Sobol results saved to results/sobol_results.json")
